In [0]:
# Databricks notebook source
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

LANDING = "/Volumes/jarvis_demo_7405604983246276/stock/raw/landing"

def autoload(subdir, schema=None):
    r = (spark.readStream.format("cloudFiles")
              .option("cloudFiles.format", "json")
              .option("cloudFiles.inferColumnTypes", "false"))
    if schema:
        r = r.schema(schema)
    return (r.load(f"{LANDING}/{subdir}")
             .withColumn("_source_file", F.col("_metadata.file_path"))
             .withColumn("_ingested_at", F.current_timestamp()))

# COMMAND ----------
# ---------------------------- BRONZE ----------------------------
# Raw, all-string, append-only. No business logic here.

PRICE_SCHEMA = StructType([
    StructField("symbol",     StringType()), StructField("trade_date", StringType()),
    StructField("run_date",   StringType()), StructField("open",       StringType()),
    StructField("high",       StringType()), StructField("low",        StringType()),
    StructField("close",      StringType()), StructField("volume",     StringType()),
])

@dlt.table(
    name="bronze_daily_prices",
    comment="Raw OHLCV from Alpha Vantage TIME_SERIES_DAILY. Overlapping 100-day windows; dedup happens in silver.",
    table_properties={"quality": "bronze"})
def bronze_daily_prices():
    return autoload("daily_prices", PRICE_SCHEMA)

@dlt.table(
    name="bronze_quote",
    comment="Raw latest-quote snapshot from GLOBAL_QUOTE, one row per symbol per run.",
    table_properties={"quality": "bronze"})
def bronze_quote():
    return autoload("quote")

@dlt.table(
    name="bronze_company",
    comment="Raw company fundamentals/profile from OVERVIEW.",
    table_properties={"quality": "bronze"})
def bronze_company():
    return autoload("company")

# COMMAND ----------
# ---------------------------- SILVER ----------------------------
# Typed, validated, deduplicated.

@dlt.view(name="v_prices_typed")
@dlt.expect_or_drop("symbol_present",  "symbol IS NOT NULL")
@dlt.expect_or_drop("date_parseable",  "trade_date IS NOT NULL")
@dlt.expect_or_drop("close_positive",  "close_price > 0")
@dlt.expect("high_ge_low",             "high_price >= low_price")
@dlt.expect("volume_non_negative",     "volume >= 0")
def v_prices_typed():
    return (dlt.read_stream("bronze_daily_prices").select(
        F.upper(F.col("symbol")).alias("symbol"),
        F.to_date("trade_date").alias("trade_date"),
        F.col("open").cast("decimal(18,4)").alias("open_price"),
        F.col("high").cast("decimal(18,4)").alias("high_price"),
        F.col("low").cast("decimal(18,4)").alias("low_price"),
        F.col("close").cast("decimal(18,4)").alias("close_price"),
        F.col("volume").cast("long").alias("volume"),
        F.to_date("run_date").alias("run_date"),
        F.col("_ingested_at"),
    ))

# Overlapping API windows mean the same (symbol, trade_date) arrives every day.
# AUTO CDC / SCD Type 1 upserts on the natural key -> one row per trading day.
dlt.create_streaming_table(
    name="silver_daily_prices",
    comment="One row per symbol per trading day. Deduplicated via SCD Type 1 upsert.",
    table_properties={"quality": "silver"})

dlt.apply_changes(
    target="silver_daily_prices",
    source="v_prices_typed",
    keys=["symbol", "trade_date"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)

# COMMAND ----------

@dlt.view(name="v_quote_typed")
@dlt.expect_or_drop("symbol_present", "symbol IS NOT NULL")
@dlt.expect_or_drop("price_positive", "price > 0")
def v_quote_typed():
    return (dlt.read_stream("bronze_quote").select(
        F.upper(F.col("symbol")).alias("symbol"),
        F.to_date("latest_trading_day").alias("latest_trading_day"),
        F.col("open").cast("decimal(18,4)").alias("open_price"),
        F.col("high").cast("decimal(18,4)").alias("high_price"),
        F.col("low").cast("decimal(18,4)").alias("low_price"),
        F.col("price").cast("decimal(18,4)").alias("price"),
        F.col("previous_close").cast("decimal(18,4)").alias("previous_close"),
        F.col("change").cast("decimal(18,4)").alias("change"),
        # strip the trailing '%' — same pattern as the '$' strip in the ETL ticket
        F.regexp_replace(F.col("change_percent"), "%", "").cast("decimal(18,4)")
         .alias("change_percent"),
        F.col("volume").cast("long").alias("volume"),
        F.col("_ingested_at"),
    ))

dlt.create_streaming_table(
    name="silver_quote_current",
    comment="Current quote per symbol. SCD Type 1 — history lives in silver_daily_prices.",
    table_properties={"quality": "silver"})

dlt.apply_changes(
    target="silver_quote_current",
    source="v_quote_typed",
    keys=["symbol"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)

# COMMAND ----------

@dlt.view(name="v_company_typed")
@dlt.expect_or_drop("symbol_present", "symbol IS NOT NULL")
def v_company_typed():
    return (dlt.read_stream("bronze_company").select(
        F.upper(F.col("symbol")).alias("symbol"),
        F.col("name").alias("company_name"),
        F.col("exchange"), F.col("currency"), F.col("country"),
        F.col("sector"), F.col("industry"),
        F.col("marketcapitalization").cast("decimal(38,2)").alias("market_cap"),
        F.col("peratio").cast("decimal(18,4)").alias("pe_ratio"),
        F.col("dividendyield").cast("decimal(18,6)").alias("dividend_yield"),
        F.col("52weekhigh").cast("decimal(18,4)").alias("week_52_high"),
        F.col("52weeklow").cast("decimal(18,4)").alias("week_52_low"),
        F.col("_ingested_at"),
    ))

# SCD Type 2 so a sector reclassification or rename is auditable.
# Volatile daily metrics are excluded from change detection, otherwise every
# run would open a new version and the dimension would grow forever.
dlt.create_streaming_table(
    name="silver_company_history",
    comment="Company dimension, SCD Type 2 on descriptive attributes only.",
    table_properties={"quality": "silver"})

dlt.apply_changes(
    target="silver_company_history",
    source="v_company_typed",
    keys=["symbol"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=2,
    except_column_list=["market_cap", "pe_ratio", "dividend_yield",
                        "week_52_high", "week_52_low", "_ingested_at"],
)

# COMMAND ----------
# ---------------------------- GOLD ----------------------------
# Materialized views: full recompute, correct window functions, cheap at this volume.

WINDOWS = [7, 30, 90]

@dlt.table(
    name="gold_price_trends",
    comment="Daily absolute and percentage price change over 7/30/90 trading days.",
    table_properties={"quality": "gold"})
def gold_price_trends():
    df = dlt.read("silver_daily_prices")
    w = Window.partitionBy("symbol").orderBy("trade_date")
    out = df.select("symbol", "trade_date", "open_price", "high_price",
                    "low_price", "close_price", "volume")
    for n in WINDOWS:
        prev = F.lag("close_price", n).over(w)
        out = (out
            .withColumn(f"close_{n}d_ago", prev)
            .withColumn(f"price_change_{n}d", F.round(F.col("close_price") - prev, 4))
            .withColumn(f"price_pct_change_{n}d",
                        F.round((F.col("close_price") - prev) / prev * 100, 4)))
    return out.withColumn("daily_change",
                          F.round(F.col("close_price") - F.lag("close_price", 1).over(w), 4))

@dlt.table(
    name="gold_volume_trends",
    comment="Rolling average/total volume over 7/30/90 trading days plus relative volume.",
    table_properties={"quality": "gold"})
def gold_volume_trends():
    df = dlt.read("silver_daily_prices")
    base = Window.partitionBy("symbol").orderBy("trade_date")
    out = df.select("symbol", "trade_date", "volume")
    for n in WINDOWS:
        w = base.rowsBetween(-(n - 1), 0)
        out = (out
            .withColumn(f"avg_volume_{n}d", F.round(F.avg("volume").over(w), 0).cast("long"))
            .withColumn(f"total_volume_{n}d", F.sum("volume").over(w))
            .withColumn(f"rel_volume_{n}d",
                        F.round(F.col("volume") / F.avg("volume").over(w), 4)))
    return out

@dlt.table(
    name="gold_ticker_snapshot",
    comment="One row per symbol: current quote + company profile + latest trend metrics. Dashboard header.",
    table_properties={"quality": "gold"})
def gold_ticker_snapshot():
    q = dlt.read("silver_quote_current")
    c = (dlt.read("silver_company_history")
            .filter(F.col("__END_AT").isNull())
            .drop("__START_AT", "__END_AT"))
    latest = (dlt.read("gold_price_trends")
                .withColumn("rn", F.row_number().over(
                    Window.partitionBy("symbol").orderBy(F.col("trade_date").desc())))
                .filter("rn = 1").drop("rn"))
    v = (dlt.read("gold_volume_trends")
            .withColumn("rn", F.row_number().over(
                Window.partitionBy("symbol").orderBy(F.col("trade_date").desc())))
            .filter("rn = 1").select("symbol", "avg_volume_30d", "rel_volume_7d"))

    return (q.join(c, "symbol", "left")
             .join(latest.select("symbol",
                                 *[f"price_pct_change_{n}d" for n in WINDOWS]),
                   "symbol", "left")
             .join(v, "symbol", "left"))

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-5616898368404110>, line 2
      1 # Databricks notebook source
----> 2 import dlt
      3 from pyspark.sql import functions as F
      4 from pyspark.sql.window import Window

ModuleNotFoundError: No module named 'dlt'